In [1]:
# Checks GPU status
!nvidia-smi

Tue Jul 22 14:30:15 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:31:00.0 Off |                   On |
| N/A   24C    P0             46W /  400W |     213MiB /  40960MiB |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [2]:
# Import packages
import json
import numpy as np
import pandas as pd 

In [3]:
# Loads the sample names of foods in the VCF dataset
vcf_path = "data/Matrix.csv"
category_path = "data/edited_category_data.json"

def get_vcf_dataframe(vcf_path, category_path):
    """Obtains a dataframe for the VCF dataset."""
    vcf_data = pd.read_csv(vcf_path)
    mixnames = vcf_data.columns[1:].values
    parsed_mixnames = ['Unnamed: 0']
    for idx, food_name in enumerate(mixnames):
        parsed_string = food_name.replace(".html", "").replace("_", " ")
        for digit in "0123456789":
            parsed_string = parsed_string.replace(digit, "")
        parsed_mixnames.append(parsed_string.strip())
    vcf_data.columns = parsed_mixnames

    # Returns binary data for each food.
    with open(category_path, "r") as file:
        category_data = json.load(file)
    combined_odors = []
    for food_category in list(category_data.keys()):
        for food in category_data[food_category]:
            if len(category_data[food_category][food]):
                for sample in category_data[food_category][food]:
                    combined_odors.append(sample)

    combined_df = vcf_data[combined_odors].T
    return combined_df

vcf_df = get_vcf_dataframe(vcf_path, category_path)
print(vcf_df)


                                          0     1     2     3     4     5     \
BANTU BEER                                 0.0   0.0   0.0   0.0   0.0   0.0   
AMERICAN BEER                              0.0   0.0   0.0   0.0   0.0   0.0   
AMERICAN BEER (premium)                    0.0   0.0   0.0   0.0   0.0   0.0   
AUSTRIAN BEER                              0.0   0.0   0.0   0.0   0.0   0.0   
CHERRY BEER (Belgian)                      0.0   0.0   0.0   0.0   0.0   0.0   
...                                        ...   ...   ...   ...   ...   ...   
RED OAK SMOKE                              0.0   0.0   0.0   0.0   0.0   0.0   
REDWOOD TREE SMOKE                         0.0   0.0   0.0   0.0   0.0   0.0   
WALNUT TREE SMOKE                          0.0   0.0   0.0   0.0   0.0   0.0   
WHITE OAK SMOKE                            0.0   0.0   0.0   0.0   0.0   0.0   
YEAST EXTRACT (saccharomyces cerevisiae)   1.0   0.0   0.0   0.0   0.0   0.0   

                                       

In [11]:
# Objective: use an LLM to obtain ten-adjective descriptions of food words in the VCF Dataset.
import requests

response = requests.post('http://localhost:8000/v1/completions', json = {
    "model": "/n/holylfs06/LABS/kempner_shared/Everyone/testbed/models/Llama-3.1-405B",
    "prompt": "San Francisco is a",
    "max_tokens": 500,
    "temperature": 0
})





ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /v1/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1532a3f237c0>: Failed to establish a new connection: [Errno 111] Connection refused'))